# L15 demo: what a datasheet costs, and what a log book clusters into

Two halves.

**Part 1** takes a pump datasheet apart with a tokenizer and asks what it costs.
The three things to watch for:

1. `Ø25` is three tokens under `cl100k_base`, and the first two are not characters.
   They are halves of one.
2. `1500` tokenizes as `150` + `0`. The model never sees the number.
3. Two models *from the same vendor* disagree about the same document by more than
   either disagrees with the other vendor.

**Part 2** embeds thirty-four maintenance log entries and asks what cosine
similarity knows. Before we compute anything you will be asked to predict which
pairs cluster. Most rooms get the near-duplicates right and the negations exactly
backwards.

**Requirements.** `tiktoken`, `numpy`, `scikit-learn`, `matplotlib`. The parts that
call a provider are guarded: without `ANTHROPIC_API_KEY` and `OPENAI_API_KEY` the
notebook still runs top to bottom and tells you which cells it skipped.

## Run this first on Colab

Colab starts from its own preinstalled environment rather than this course's `uv`
environment, so run the cell below before anything else. It installs what this
notebook needs and Colab does not already have. Outside Colab it does nothing, so
you can run it or skip it.


In [ ]:
# Run this first on Colab. Anywhere else this cell does nothing.
#
# Only genuinely missing packages are installed, so Colab's own versions of
# everything it already ships are left alone.
#
# Generated by tools/colab_setup.py from this notebook's imports. Edit that.
import importlib.util
import subprocess
import sys

REQUIREMENTS = {
    "numpy": "numpy",
    "openai": "openai",
    "sklearn": "scikit-learn",
    "tiktoken": "tiktoken",
}


def _missing(module):
    try:
        return importlib.util.find_spec(module) is None
    except ModuleNotFoundError:  # the parent package is absent
        return True


if "google.colab" in sys.modules:
    need = sorted({pip for mod, pip in REQUIREMENTS.items() if _missing(mod)})
    if need:
        print("installing:", " ".join(need))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
    print("Colab setup done." if need else "Colab: nothing to install.")


In [ ]:
import json
import os
import urllib.request

import numpy as np
import tiktoken

HAVE_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
HAVE_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
print(f"ANTHROPIC_API_KEY set: {HAVE_ANTHROPIC}")
print(f"OPENAI_API_KEY set:    {HAVE_OPENAI}")

CL100K = tiktoken.get_encoding("cl100k_base")   # GPT-3.5/4 era
O200K = tiktoken.get_encoding("o200k_base")     # GPT-4o era
print(f"\ncl100k_base vocabulary: {CL100K.n_vocab:,} tokens")
print(f"o200k_base  vocabulary: {O200K.n_vocab:,} tokens")

## Part 1, a datasheet

This is a pump specification written for this course rather than copied from a
vendor, so that it can be redistributed. Every convention in it is real: ASME
flange classes, AISI and ASTM material designations, ISO fits, IP ratings.

Read it the way a tokenizer will have to.

In [ ]:
DATASHEET = """\
CENTRIFUGAL PROCESS PUMP - MODEL CP-4L/2200-XG
Document 4L-2200-XG-DS Rev. C

1. GENERAL
Single-stage, end-suction centrifugal pump conforming to ASME B73.1. Foot-mounted,
back pull-out design. Nominal capacity 2200 L/min at 45 m total dynamic head.
Maximum working pressure 10.5 MPa (1523 psig) at 20 degC. Operating temperature
range -40 degC to +120 degC. Hydrostatic test pressure 15.8 MPa held for 30 min.

2. MATERIALS OF CONSTRUCTION
Casing: ASTM A216 WCB carbon steel. Impeller: SS316L, investment cast, Ra 0.8 um
finish on wetted surfaces. Shaft: AISI 4140 quenched and tempered, 28-32 HRC.
Shaft sleeve: Alloy 20. Wear rings: Ni-resist Type 2, replaceable. Gaskets: PTFE
envelope with aramid filler. Fasteners: 1/4-20 UNC and M8x1.25 per ISO 898-1
Class 8.8.

3. DIMENSIONS AND TOLERANCES
Suction NPS 4 Class 300 RF flange per ASME B16.5. Discharge NPS 2 Class 300 RF.
Impeller bore diameter 25 mm +0.021/-0.000 (H7). Shaft diameter at coupling
25.000 mm +/-0.005 mm. Bearing housing bore 72.000 mm H7. Axial float 0.0015 in
maximum. Impeller trim range 178 mm to 210 mm; supplied trim 197 mm unless
otherwise specified on the order.

4. DRIVER AND ELECTRICALS
Motor: 3.5 kW, 3-phase, 460 V, 60 Hz, 1750 rpm nominal, TEFC, IP66, IE3 premium
efficiency. Full-load current 5.8 A. Service factor 1.15. Motor frame 132M per
IEC 60072. Terminal box torque 4.5 N-m.

5. PERFORMANCE
Best efficiency point 2200 L/min at 45 m, efficiency 78 percent, NPSHr 3.2 m.
Shutoff head 52 m. Minimum continuous stable flow 660 L/min. Flow coefficient
Cv 12.4 for the bypass control valve supplied with the skid. Vibration limit
4.5 mm/s RMS per ISO 10816-3 measured at the bearing housing.

6. INSTRUMENTATION
Bearing temperature: two K-type thermocouples, one per bearing, 4-20 mA
transmitters, range 0-150 degC, accuracy +/-0.5 percent of span. Discharge
pressure: piezoresistive transmitter, range 0-16 MPa, accuracy +/-0.25 percent
FS. Suction pressure: 0-2.5 MPa, same accuracy class.

7. SEALING
Single mechanical seal per API 682 Category 1, Type A, Arrangement 1, Plan 11
flush. Seal faces: sintered SiC vs carbon graphite. Elastomers: FKM, -20 degC to
+200 degC. Maximum seal chamber pressure 2.5 MPa.

8. MASS AND SHIPPING
Bare pump mass 148 kg. Baseplate 96 kg. Motor 62 kg. Total shipping mass 341 kg
including crate. Crate dimensions 1400 mm x 800 mm x 900 mm.
"""

print(DATASHEET[:380])
print("...")
print(f"\n{len(DATASHEET):,} characters, {len(DATASHEET.split()):,} whitespace-separated words")

### Predict first

Before running the next cell, write down your guess for how many tokens each of
these becomes under `cl100k_base`:

| string | characters | your guess |
|---|---|---|
| `10.5 MPa` | 8 | |
| `AISI 4140` | 9 | |
| `Ø25` | 3 | |
| `P/N 4L-2200-XG` | 14 | |

The usual room average is about one token per three characters. Two of these are
much worse than that.

In [ ]:
def show(s, enc=CL100K, name="cl100k_base"):
    ids = enc.encode(s)
    pieces = [enc.decode([i]) for i in ids]
    shown = " | ".join(repr(p)[1:-1] for p in pieces)
    print(f"{s!r:22s} {len(s):3d} chars -> {len(ids):2d} tokens   {shown}")


for s in ["10.5 MPa", "AISI 4140", "Ø25", "±0.05 mm",
          "P/N 4L-2200-XG", "1500 rpm", "SS316L", "Ra 0.8 µm",
          "NPS 4 Class 300", "1/4-20 UNC"]:
    show(s)

### The `Ø25` surprise

`Ø25` came back as three tokens, and if you look at the printed fragments the
first two are replacement characters. That is not a display bug. `Ø` is two bytes
in UTF-8, and `cl100k_base` has no merge rule for that byte pair, so the encoder
emitted each byte as its own token. **Neither one decodes to a character.**

Under the newer `o200k_base` the same string is two tokens. Nothing about the
document changed.

In [ ]:
s = "Ø25"
print(f"{s!r} is {len(s)} characters and {len(s.encode('utf-8'))} UTF-8 bytes: "
      f"{list(s.encode('utf-8'))}")
print()
show(s, CL100K, "cl100k_base")
show(s, O200K, "o200k_base")
print()
print("the individual token ids and what each one decodes to:")
for i in CL100K.encode(s):
    raw = CL100K.decode_single_token_bytes(i)
    print(f"  id {i:6d}  bytes {list(raw)}  decodes to {raw.decode('utf-8', 'replace')!r}")

### Numbers do not tokenize where you think

Digits are grouped in runs of up to three, left to right, regardless of what the
number means. A four-digit number becomes a three-digit chunk and a leftover.

In [ ]:
for s in ["10", "105", "1050", "1500", "4140", "2200", "10.5", "0.05", "12.4"]:
    ids = CL100K.encode(s)
    print(f"{s!r:8s} -> {[CL100K.decode([i]) for i in ids]}")

This is a large part of why LLMs are unreliable at arithmetic on long numbers: the
representation itself does not respect place value. It is also why a value like
`0.0015` is fragile in an extraction pipeline. The model has to reassemble it from
pieces that carry no numeric meaning individually.

### Case and whitespace are not free either

In [ ]:
for s in ["bearing", " bearing", "Bearing", "BEARING", "bearing.", "  bearing"]:
    ids = CL100K.encode(s)
    print(f"{s!r:12s} -> {len(ids)} token(s): {[CL100K.decode([i]) for i in ids]}")

Maintenance logs are usually written in capitals. That costs more tokens *and*
presents the model with different symbols than the lower-case training text.

### What it adds up to per document

The number worth carrying in your head is characters per token. The rule of thumb
everyone quotes, about four characters per token, is a fact about English prose.

In [ ]:
PROSE = """\
The pump should be started against a partly closed discharge valve and brought up
to speed before the valve is opened. Opening the valve too quickly can drive the
operating point far to the right of the best efficiency point, where the radial
load on the impeller rises sharply and the shaft deflects enough to shorten seal
life. Whenever the machine is going to be idle for an extended period, the casing
should be drained and the shaft rotated by hand at intervals.
"""

TABLE = """\
Tag  Service          Set point  Units  Lo-Lo  Lo     Hi     Hi-Hi
PT-101  Discharge     8.5        MPa    2.0    4.0    9.5    10.2
PT-102  Suction       0.35       MPa    0.10   0.15   1.80   2.20
TT-201  Brg NDE       65         degC   -      -      85     95
FT-301  Flow          2200       L/min  660    900    2600   2900
VT-401  Vibration     2.8        mm/s   -      -      4.5    7.1
"""

CODE = """\
def npsha(p_suction_pa, p_vapor_pa, rho, v_ms, z_m, g=9.80665):
    static = (p_suction_pa - p_vapor_pa) / (rho * g)
    return static + v_ms**2 / (2 * g) + z_m
"""

print(f"{'text':16s} {'chars':>7s} {'tokens':>7s} {'chars/token':>12s}")
for label, text in [("technical prose", PROSE), ("pump datasheet", DATASHEET),
                    ("alarm table", TABLE), ("python code", CODE)]:
    n = len(CL100K.encode(text))
    print(f"{label:16s} {len(text):7,d} {n:7,d} {len(text)/n:12.2f}")

A page of a datasheet costs roughly 1.6 times as many tokens as a page of prose
with the same number of characters. A page of tabular data costs about twice.

### Now count it the way you will be billed

`tiktoken` is a local library, so counting is free and instant. It is also exact
only for OpenAI models. Anthropic publishes no offline tokenizer; it exposes a
[token counting endpoint](https://platform.claude.com/docs/en/build-with-claude/token-counting)
instead, which is free to call but is a network round trip against its own rate
limit.

In [ ]:
def anthropic_count(model, text):
    """Tokens as the provider counts them, for the model you will actually call."""
    body = json.dumps({"model": model,
                       "messages": [{"role": "user", "content": text}]}).encode()
    req = urllib.request.Request(
        "https://api.anthropic.com/v1/messages/count_tokens",
        data=body,
        headers={"x-api-key": os.environ["ANTHROPIC_API_KEY"],
                 "anthropic-version": "2023-06-01",
                 "content-type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        return json.load(resp)["input_tokens"]


counts = {"cl100k_base (OpenAI)": len(CL100K.encode(DATASHEET)),
          "o200k_base (OpenAI)": len(O200K.encode(DATASHEET))}

if HAVE_ANTHROPIC:
    for model in ["claude-haiku-4-5", "claude-opus-5"]:
        counts[model] = anthropic_count(model, DATASHEET)
else:
    print("[no ANTHROPIC_API_KEY: showing the counts measured on 2026-08-06]")
    counts["claude-haiku-4-5"] = 924
    counts["claude-opus-5"] = 1263

base = counts["cl100k_base (OpenAI)"]
print(f"\nthe same {len(DATASHEET):,} characters:\n")
for name, n in counts.items():
    rel = "baseline" if n == base else f"{n/base - 1:+.0%}"
    print(f"  {name:24s} {n:6,d} tokens   {rel}")

pair = counts["claude-opus-5"] / counts["claude-haiku-4-5"] - 1
print(f"\ntwo models from the SAME vendor disagree by {pair:+.0%}")

That last line is the one to take away. "Use the provider's tokenizer" is not a
sufficient rule, because the tokenizer changed inside the vendor's own model line.
Use the *model's*, for the exact model id you are going to call.

### The budget arithmetic

Three numbers you should compute for one representative document before you build
anything: tokens, dollars, seconds. Here are the first two. Prices are US dollars
per million input tokens, read from the providers' pricing pages on 2026-08-06,
and they change.

In [ ]:
PRICES = {"claude-opus-5": 5.00, "claude-haiku-4-5": 1.00}
N_DOCS = 500          # a modest corpus
QUESTIONS_PER_DOC = 3

print(f"{'model':20s} {'tok/doc':>9s} {'$/call':>9s} {'$/corpus':>10s} {'$ cached':>10s}")
for model, price in PRICES.items():
    n = counts[model]
    per_call = n * price / 1e6
    corpus = per_call * N_DOCS * QUESTIONS_PER_DOC
    # a cache read is roughly a tenth of the input price; the first call still pays
    cached = per_call * N_DOCS * (1 + 0.1 * (QUESTIONS_PER_DOC - 1))
    print(f"{model:20s} {n:9,d} {per_call:9.5f} {corpus:10.2f} {cached:10.2f}")

print(f"\n{N_DOCS} documents, {QUESTIONS_PER_DOC} questions each.")
print("The 'cached' column assumes prompt caching, which most providers now offer:")
print("the first call writes the cache, the rest read it at about a tenth the price.")

## Part 2, a log book

Thirty-four free-text maintenance entries, written the way maintenance entries are
actually written. There are deliberate structures in here:

- five different ways of reporting the same bearing noise
- four entries that *negate* an earlier one ("no leak found")
- two pairs that differ only in the unit

In [ ]:
LOGS = [
    "Bearing noise on pump P-101 drive end during startup",
    "Noisy bearing, P-101 DE, audible at start",
    "brg vibration p101 high at startup",
    "Operator reports growling from the drive end bearing on P-101",
    "P-101 DE bearing running rough on morning start",
    "Mechanical seal leaking, approx 8 drops/min, pump P-101",
    "Seal drip observed at P-101 seal chamber, catch pan wet",
    "P-101 mech seal weeping, no visible spray",
    "Leak from the seal gland on P-101, product on the baseplate",
    "Motor M-101 winding temperature high, 118 degC at full load",
    "M-101 running hot, winding RTD reading 118 C",
    "Overtemperature on pump motor M-101 under load",
    "Coupling guard loose, fasteners backed out on P-101",
    "Misalignment suspected P-101, laser check requested",
    "Shaft alignment out of tolerance after grout repair, P-101",
    "PT-101 reading 0.0 MPa with pump running, transmitter suspect",
    "Discharge pressure transmitter appears failed low on P-101",
    "TT-201 thermocouple open circuit, reading drove to upscale burnout",
    "P-101 seal inspected, no leak found",
    "Bearing checked, no abnormal noise detected on P-101",
    "P-101 DE bearing replaced, vibration back to 1.8 mm/s",
    "Motor M-101 temperature normal after fan cowl cleaned",
    "Strainer differential 45 kPa, basket cleaned and refitted",
    "Impeller wear ring clearance measured 0.55 mm, above 0.40 mm limit",
    "Baseplate grout cracked at the northeast anchor bolt",
    "Suction line support hanger found detached from the pipe rack",
    "Lube oil sample taken, ISO 4406 code 20/18/15, filter changed",
    "Spare pump P-102 exercised for 30 min, no findings",
    "Painted casing where insulation had been removed for inspection",
    "Annual relief valve PSV-110 bench tested, set 11.0 MPa, passed",
    "Discharge pressure trending at 10.5 MPa, near the 10.2 MPa Hi-Hi",
    "Discharge pressure trending at 10.5 bar, well below the trip",
    "Bearing temperature 85 degC steady on the drive end",
    "Bearing temperature 85 degF steady on the drive end",
]
print(f"{len(LOGS)} entries, {sum(len(x) for x in LOGS):,} characters")
print(f"{len(CL100K.encode(chr(10).join(LOGS))):,} tokens to embed the lot")

### Predict first, again

Which of these three pairs do you expect to score highest on cosine similarity?

**A.** `brg vibration p101 high at startup`
    vs `Operator reports growling from the drive end bearing on P-101`

**B.** `Mechanical seal leaking, approx 8 drops/min, pump P-101`
    vs `P-101 seal inspected, no leak found`

**C.** `Bearing temperature 85 degC steady on the drive end`
    vs `Bearing temperature 85 degF steady on the drive end`

A describes the same event twice. B describes an event and its refutation. C
differs by one letter and a factor of about two in temperature.

Write down your ranking before running the next cells.

In [ ]:
def unit_rows(v):
    return v / np.linalg.norm(v, axis=1, keepdims=True)


EMBED_MODEL = "text-embedding-3-small"

if HAVE_OPENAI:
    from openai import OpenAI

    resp = OpenAI().embeddings.create(model=EMBED_MODEL, input=LOGS)
    E = unit_rows(np.array([d.embedding for d in resp.data], dtype=np.float64))
    print(f"embedded {len(LOGS)} entries in ONE call, "
          f"{resp.usage.total_tokens} tokens, {E.shape[1]} dimensions")
else:
    print("[no OPENAI_API_KEY: falling back to a lexical baseline so the rest runs]")
    print("[the numbers below will NOT match the notes: TF-IDF is not an embedding]")
    from sklearn.feature_extraction.text import TfidfVectorizer

    E = unit_rows(np.asarray(TfidfVectorizer().fit_transform(LOGS).todense()))

S = E @ E.T          # vectors are unit length, so this is cosine similarity
print(f"similarity matrix: {S.shape}, computed in one matrix multiply")

Note what just happened to the cost. Comparing 34 entries pairwise with an LLM is
561 calls. Embedding them was **one** call and a few hundred tokens, after which
every pairwise comparison is a dot product. At a hundred thousand records the LLM
approach is not expensive, it is arithmetically impossible.

### The three pairs

In [ ]:
def idx(fragment):
    return next(i for i, t in enumerate(LOGS) if fragment in t)


# measured with text-embedding-3-small on 2026-08-06, for the no-key path
REFERENCE = {"A": 0.532, "B": 0.694, "C": 0.971}

PAIRS = [
    ("A  same event", idx("brg vibration"), idx("growling")),
    ("B  the opposite", idx("Mechanical seal leaking"), idx("no leak found")),
    ("C  unit swap", idx("85 degC"), idx("85 degF")),
]
for label, i, j in PAIRS:
    ref = REFERENCE[label[0]]
    note = "" if HAVE_OPENAI else f"   (embedding cosine was {ref:.3f})"
    print(f"{label:18s} {S[i, j]:.3f}{note}")
    print(f"{'':18s} {LOGS[i]}")
    print(f"{'':18s} {LOGS[j]}\n")

print("B, which means the OPPOSITE, scores higher than A, which means the SAME.")

### Against a lexical baseline

The argument for embeddings is the pairs that share no words at all.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

T = unit_rows(np.asarray(TfidfVectorizer().fit_transform(LOGS).todense()))
L = T @ T.T

i, j = idx("brg vibration"), idx("growling")
print(f"{LOGS[i]}\n{LOGS[j]}\n")
print(f"  TF-IDF cosine (words in common): {L[i, j]:.3f}")
print(f"  embedding cosine:                {S[i, j]:.3f}")
print("\nEvery keyword search, every LIKE '%bearing%', misses this pair.")

### And now the part that should bother you

The bearing-noise cluster and its negations, side by side.

In [ ]:
SAME = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (5, 6), (5, 7),
        (6, 8), (9, 10), (9, 11), (13, 14), (15, 16)]
OPPOSITE = [(5, 18), (0, 19), (1, 19), (0, 20), (2, 20), (9, 21), (30, 31), (32, 33)]

same = np.array([S[i, j] for i, j in SAME])
opp = np.array([S[i, j] for i, j in OPPOSITE])

print(f"pairs that mean the SAME:     min {same.min():.3f}  median {np.median(same):.3f}"
      f"  max {same.max():.3f}")
print(f"pairs that mean the OPPOSITE: min {opp.min():.3f}  median {np.median(opp):.3f}"
      f"  max {opp.max():.3f}")
print()
n_above = int((opp > same.min()).sum())
print(f"{n_above} of {len(opp)} opposite-meaning pairs score ABOVE the weakest true match")

Cosine similarity measures how much two texts are *about the same thing*. A report
of a leak and a report of no leak are maximally about the same thing. The word
"no" is one token among a dozen and it does not move the vector far.

Nothing in the training objective of an embedding model requires it to.

### So pick a threshold

In [ ]:
print(f"{'cut':>6s} {'recall':>8s} {'precision':>10s}   admits")
for t in [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]:
    tp = int((same >= t).sum())
    fp = int((opp >= t).sum())
    rec = tp / len(same)
    prec = tp / (tp + fp) if tp + fp else float("nan")
    print(f"{t:6.2f} {rec:8.2f} {prec:10.2f}   {fp} of {len(opp)} opposite pairs")

print("\nThere is no cut that keeps the true matches and rejects the negations.")
print("Not a badly chosen one. There is no value, because the classes overlap.")

### Dimensionality is a storage decision

A million chunks at 1,536 float32 dimensions is 6 GB before any index overhead.
Both OpenAI's v3 models and Voyage's current models are trained so that the vector
can be truncated from the end and renormalized.

This cell needs an API key; without one it prints the values measured on
2026-08-06.

In [ ]:
DIMS = [1536, 1024, 512, 256, 128, 64]
MEASURED = {1536: 1.000, 1024: 0.971, 512: 1.000, 256: 0.853, 128: 0.824, 64: 0.676}

nn_full = [int(np.argsort(-S[i])[1]) for i in range(len(LOGS))]

print(f"{'dims':>6s} {'bytes/vec':>10s} {'top-1 neighbour unchanged':>26s}")
for d in DIMS:
    if HAVE_OPENAI:
        r = OpenAI().embeddings.create(model=EMBED_MODEL, input=LOGS, dimensions=d)
        Ed = unit_rows(np.array([x.embedding for x in r.data], dtype=np.float64))
        Sd = Ed @ Ed.T
        nn_d = [int(np.argsort(-Sd[i])[1]) for i in range(len(LOGS))]
        agree = float(np.mean([a == b for a, b in zip(nn_full, nn_d)]))
    else:
        agree = MEASURED[d]
    print(f"{d:6d} {d*4:10,d} {agree:25.1%}")

print("\nRead that honestly: 1024 does WORSE than 512. The curve is not monotonic,")
print("because 34 records is not a sample. Tuning storage on this would be tuning")
print("on noise.")

## What to carry into A8

Three habits, all of which you just measured:

**Count the tokens with the model you are going to call.** Not with the library you
happen to have installed. A8 task 2 asks for a token and cost baseline over your
corpus, and you can do that today, before writing any extraction code.

**Ground the question and give the model a way to say no.** The escape-hatch
measurement in the notes is the whole reason A8 requires a "field not present"
path. A missing value must be a legal answer, in the prompt *and* in the schema.

**Never let semantic similarity decide a unit.** `85 degC` and `85 degF` scored
0.971. Units belong in the schema, and normalization belongs in an explicit,
tested pipeline stage that keeps the original value alongside the converted one.